# 17 — Enhancement Evaluation

Evaluate the three condition-specific enhancement models trained in Notebooks 14–16:

- **Night** → `models/enhancement/night/night_enhancer_best.pth`
- **Rain** → `models/enhancement/rain/rain_enhancer_best.pth`
- **Fog** → `models/enhancement/fog/fog_enhancer_best.pth`

The evaluation is condition-aware and keeps the original BDD100K validation/test splits separated. It reports reconstruction-style metrics against the same deterministic pseudo-reference definitions used during training, plus image-quality changes and visual comparisons.

Important: pseudo-reference metrics are not photographic ground truth. Later downstream detection/tracking evaluation is required to judge whether enhancement helps the complete Adaptive_Road_Vision system.

In [ ]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *list(current.parents)]
    for candidate in candidates:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback
    return current

PROJECT_ROOT = find_project_root()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. "
        "Edit PROJECT_ROOT to your Adaptive_Road_Vision folder."
    )

def resolve_dataset_root():
    candidates = [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]
    for candidate in candidates:
        if (candidate / "metadata" / "final_dataset_metadata.csv").exists():
            return candidate
        if (candidate / "final_dataset_metadata.csv").exists():
            return candidate
    return candidates[0]

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
MODEL_DIR = PROJECT_ROOT / "models" / "enhancement"
RESULTS_DIR = PROJECT_ROOT / "results" / "enhancement"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0
MAX_VISUALS_PER_CONDITION = 4

random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", FINAL_DIR)
print("Checkpoint root:", MODEL_DIR)
print("Output directory:", RESULTS_DIR)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load final dataset metadata

In [ ]:
metadata_candidates = [
    FINAL_DIR / "metadata" / "final_dataset_metadata.csv",
    FINAL_DIR / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
metadata_path = next((p for p in metadata_candidates if p.exists()), None)

if metadata_path is None:
    raise FileNotFoundError(
        "Dataset metadata not found. Expected final_dataset_metadata.csv "
        "inside the final dataset metadata folder. Run Notebook 10 first."
    )

meta = pd.read_csv(metadata_path)
if "project_condition" not in meta.columns:
    if "condition" in meta.columns:
        meta["project_condition"] = meta["condition"]
    elif "weather" in meta.columns:
        meta["project_condition"] = meta["weather"]
    else:
        raise KeyError("Metadata needs project_condition, condition, or weather column.")

condition_col = "project_condition"
meta[condition_col] = (
    meta[condition_col].astype(str).str.strip().str.lower()
    .replace({
        "nighttime": "night", "night_time": "night",
        "rainy": "rain", "rainy_weather": "rain",
        "foggy": "fog", "foggy_weather": "fog",
    })
)

if "split" not in meta.columns:
    raise KeyError("Dataset metadata is missing the required 'split' column.")

meta["split"] = (
    meta["split"].astype(str).str.strip().str.lower()
    .replace({"validation": "val", "valid": "val", "testing": "test"})
)

def resolve_image(row):
    for col in ["final_image_path", "image_path", "resolved_path", "file_path"]:
        if col in row.index and pd.notna(row[col]):
            raw = str(row[col]).strip()
            if raw:
                p = Path(raw)
                if p.exists():
                    return p
                # Resolve relative paths against project and dataset roots.
                for base in [PROJECT_ROOT, FINAL_DIR]:
                    candidate = base / raw
                    if candidate.exists():
                        return candidate

    split = str(row.get("split", "train"))
    name = str(row.get("image_name", row.get("image_id", ""))).strip()
    if not name or name.lower() == "nan":
        return None
    if not name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
        name += ".jpg"

    # Try common image directory layouts and split aliases.
    split_aliases = {"val": ["val", "validation", "valid"], "train": ["train"], "test": ["test"]}
    for split_name in split_aliases.get(split, [split]):
        for base in [
            FINAL_DIR / "images" / split_name / name,
            FINAL_DIR / split_name / name,
            FINAL_DIR / "images" / name,
            PROJECT_ROOT / "datasets" / "images" / split_name / name,
        ]:
            if base.exists():
                return base
    return None

meta["resolved_path"] = meta.apply(resolve_image, axis=1)
missing_paths = int(meta["resolved_path"].isna().sum())
meta = meta[meta["resolved_path"].notna()].copy()
meta["resolved_path"] = meta["resolved_path"].map(str)

print("Metadata:", metadata_path)
print("Rows with missing/unresolved image paths:", missing_paths)
print("Usable rows:", len(meta))
print("Conditions:", meta[condition_col].value_counts().to_dict())
print("Splits:", meta["split"].value_counts().to_dict())

if meta.empty:
    raise RuntimeError("No usable image paths were resolved. Check the dataset root and metadata paths.")


## 2. Condition-specific pseudo-reference functions

These functions match the target-generation logic used by Notebooks 14–16.

- Night: exposure + contrast + gamma correction.
- Rain: conservative median/Gaussian smoothing + slight contrast recovery.
- Fog: low-frequency atmospheric veil estimation + conservative detail recovery.

The identity output is also evaluated as a baseline, so the enhancement model must be interpreted relative to doing nothing.

In [ ]:
def night_reference(img):
    img = img.convert("RGB")
    img = ImageEnhance.Brightness(img).enhance(1.55)
    img = ImageEnhance.Contrast(img).enhance(1.12)

    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.power(
        np.clip(arr, 0, 1),
        0.78
    )

    return Image.fromarray(
        np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    )


def rain_reference(img):
    img = img.convert("RGB")

    smooth = img.filter(
        ImageFilter.MedianFilter(size=3)
    )
    smooth = smooth.filter(
        ImageFilter.GaussianBlur(radius=0.35)
    )

    a = np.asarray(img).astype(np.float32)
    b = np.asarray(smooth).astype(np.float32)

    out = 0.70 * a + 0.30 * b

    out_img = Image.fromarray(
        np.clip(out, 0, 255).astype(np.uint8)
    )

    return ImageEnhance.Contrast(
        out_img
    ).enhance(1.03)


def fog_reference(img):
    img = img.convert("RGB")

    arr = (
        np.asarray(img)
        .astype(np.float32)
        / 255.0
    )

    smooth_img = img.filter(
        ImageFilter.GaussianBlur(radius=4.0)
    )

    smooth = (
        np.asarray(smooth_img)
        .astype(np.float32)
        / 255.0
    )

    detail = arr - smooth
    restored = smooth + 1.55 * detail
    restored = np.clip(
        restored,
        0.0,
        1.0
    )

    out = Image.fromarray(
        (restored * 255.0)
        .astype(np.uint8)
    )

    out = ImageEnhance.Contrast(
        out
    ).enhance(1.08)

    return ImageEnhance.Sharpness(
        out
    ).enhance(1.12)


REFERENCE_FUNCTIONS = {
    "night": night_reference,
    "rain": rain_reference,
    "fog": fog_reference,
}

## 3. Model architecture and checkpoint loading

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1
            ),
            nn.InstanceNorm2d(
                channels,
                affine=True
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1
            ),
            nn.InstanceNorm2d(
                channels,
                affine=True
            ),
        )

    def forward(self, x):
        return x + self.block(x)


class ConditionEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(
                32,
                64,
                3,
                stride=2,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(
                64,
                128,
                3,
                stride=2,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.res = nn.Sequential(
            *[
                ResidualBlock(128)
                for _ in range(4)
            ]
        )

        self.dec2 = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.dec1 = nn.Sequential(
            nn.Conv2d(
                64,
                32,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.out = nn.Conv2d(
            32,
            3,
            3,
            padding=1
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        z = self.res(e3)

        z = F.interpolate(
            z,
            size=e2.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        z = self.dec2(z) + e2

        z = F.interpolate(
            z,
            size=e1.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        z = self.dec1(z) + e1

        residual = (
            torch.tanh(
                self.out(z)
            ) * 0.5
        )

        return torch.clamp(
            x + residual,
            0,
            1
        )


CHECKPOINTS = {
    "night": MODEL_DIR / "night" / "night_enhancer_best.pth",
    "rain": MODEL_DIR / "rain" / "rain_enhancer_best.pth",
    "fog": MODEL_DIR / "fog" / "fog_enhancer_best.pth",
}

for condition, path in CHECKPOINTS.items():
    print(
        condition.upper(),
        "->",
        path,
        "| exists:",
        path.exists()
    )

In [ ]:
def load_checkpoint(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Required trained checkpoint is missing: {path}. "
            "Run the matching training notebook (14 Night, 15 Rain, or 16 Fog) first."
        )

    model = ConditionEnhancer().to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE)

    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            state = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state = checkpoint["state_dict"]
        else:
            state = checkpoint
    else:
        state = checkpoint

    # Handle DataParallel checkpoints, if present.
    if isinstance(state, dict) and state and all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


missing_checkpoints = [str(p) for p in CHECKPOINTS.values() if not p.exists()]
if missing_checkpoints:
    raise FileNotFoundError(
        "One or more trained enhancement checkpoints are missing:\n- "
        + "\n- ".join(missing_checkpoints)
        + "\nRun Notebooks 14–16 before running this evaluation."
    )

models = {condition: load_checkpoint(path) for condition, path in CHECKPOINTS.items()}
print("Loaded trained models:", list(models.keys()))


## 4. Evaluation metrics

PSNR and SSIM are calculated against the condition-specific pseudo-reference. Additional statistics measure mean brightness, contrast, and saturation before and after enhancement.

The identity baseline (`input → unchanged input`) is included so the enhancement model is not evaluated in isolation.

**Interpretation caveat:** the pseudo-reference is generated by deterministic image filters, not paired clean ground truth. PSNR/SSIM here measure similarity to that heuristic target and should not be presented as validated real-world restoration quality. Use the downstream object-detection comparison and visual inspection as complementary evidence.


In [ ]:
def gaussian_window(
    window_size=11,
    sigma=1.5,
    channels=3,
    device=None
):
    coords = (
        torch.arange(
            window_size,
            device=device
        ).float()
        - window_size // 2
    )

    g = torch.exp(
        -(coords ** 2)
        / (2 * sigma ** 2)
    )

    g = g / g.sum()

    kernel = torch.outer(
        g,
        g
    )

    return kernel.expand(
        channels,
        1,
        window_size,
        window_size
    ).contiguous()


def ssim_torch(
    x,
    y,
    window_size=11
):
    channels = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        channels,
        x.device
    )

    mu_x = F.conv2d(
        x,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_y = F.conv2d(
        y,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x = (
        F.conv2d(
            x * x,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_x2
    )

    sigma_y = (
        F.conv2d(
            y * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_y2
    )

    sigma_xy = (
        F.conv2d(
            x * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_xy
    )

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    score = (
        (2 * mu_xy + C1)
        * (2 * sigma_xy + C2)
        /
        (
            (mu_x2 + mu_y2 + C1)
            * (sigma_x + sigma_y + C2)
            + 1e-8
        )
    )

    return score.mean()


def psnr_torch(x, y):
    mse = F.mse_loss(
        x,
        y
    ).item()

    return 10 * math.log10(
        1.0 / max(mse, 1e-12)
    )


def image_stats(tensor):
    arr = (
        tensor.detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

    gray = (
        0.299 * arr[:, :, 0]
        + 0.587 * arr[:, :, 1]
        + 0.114 * arr[:, :, 2]
    )

    contrast = float(
        gray.std()
    )

    saturation = float(
        arr.max(axis=2).mean()
        - arr.min(axis=2).mean()
    )

    return {
        "brightness": float(gray.mean()),
        "contrast": contrast,
        "saturation": saturation,
    }

## 5. Condition-aware evaluation dataset

Validation and test are evaluated independently. Only the matching condition is sent through each enhancement model:

- Night → Night model
- Rain → Rain model
- Fog → Fog model

Other conditions are not passed through the condition-specific model.

In [ ]:
class EvaluationDataset(Dataset):
    def __init__(
        self,
        frame,
        reference_function
    ):
        self.frame = frame.reset_index(
            drop=True
        )
        self.reference_function = reference_function

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        path = row["resolved_path"]

        img = Image.open(
            path
        ).convert("RGB")

        img = img.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

        target = self.reference_function(
            img
        )

        x = TF.to_tensor(img)
        y = TF.to_tensor(target)

        return (
            x,
            y,
            str(path)
        )


def make_condition_frame(
    split,
    condition
):
    return meta[
        (meta["split"].astype(str).str.lower() == split)
        &
        (
            meta[condition_col]
            .astype(str)
            .str.lower()
            == condition
        )
    ].copy()


evaluation_frames = {}

for split in ["val", "test"]:
    for condition in [
        "night",
        "rain",
        "fog"
    ]:
        frame = make_condition_frame(
            split,
            condition
        )

        evaluation_frames[
            (split, condition)
        ] = frame

        print(
            f"{split:>4} | "
            f"{condition:>5} | "
            f"{len(frame):>6} images"
        )

## 6. Run quantitative evaluation

For every image, collect:

- identity PSNR/SSIM,
- enhanced PSNR/SSIM,
- PSNR/SSIM change,
- brightness/contrast/saturation before and after enhancement.

The per-image CSV makes later error analysis possible.

In [ ]:
all_rows = []

for (split, condition), frame in evaluation_frames.items():
    if len(frame) == 0:
        print(
            f"Skipping {split}/{condition}: no images."
        )
        continue

    dataset = EvaluationDataset(
        frame,
        REFERENCE_FUNCTIONS[condition]
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    model = models[condition]

    for x, target, paths in loader:
        x = x.to(
            DEVICE,
            non_blocking=True
        )

        target = target.to(
            DEVICE,
            non_blocking=True
        )

        with torch.no_grad():
            enhanced = model(x)

        for i, path in enumerate(paths):
            xi = x[i:i+1]
            yi = target[i:i+1]
            ei = enhanced[i:i+1]

            identity_psnr = psnr_torch(
                xi.float(),
                yi.float()
            )

            enhanced_psnr = psnr_torch(
                ei.float(),
                yi.float()
            )

            identity_ssim = ssim_torch(
                xi.float(),
                yi.float()
            ).item()

            enhanced_ssim = ssim_torch(
                ei.float(),
                yi.float()
            ).item()

            before = image_stats(
                x[i]
            )

            after = image_stats(
                enhanced[i]
            )

            all_rows.append({
                "split": split,
                "condition": condition,
                "image_path": path,
                "identity_psnr": identity_psnr,
                "enhanced_psnr": enhanced_psnr,
                "psnr_delta": (
                    enhanced_psnr
                    - identity_psnr
                ),
                "identity_ssim": identity_ssim,
                "enhanced_ssim": enhanced_ssim,
                "ssim_delta": (
                    enhanced_ssim
                    - identity_ssim
                ),
                "input_brightness": before[
                    "brightness"
                ],
                "enhanced_brightness": after[
                    "brightness"
                ],
                "brightness_delta": (
                    after["brightness"]
                    - before["brightness"]
                ),
                "input_contrast": before[
                    "contrast"
                ],
                "enhanced_contrast": after[
                    "contrast"
                ],
                "contrast_delta": (
                    after["contrast"]
                    - before["contrast"]
                ),
                "input_saturation": before[
                    "saturation"
                ],
                "enhanced_saturation": after[
                    "saturation"
                ],
                "saturation_delta": (
                    after["saturation"]
                    - before["saturation"]
                ),
            })

    print(
        f"Finished {split}/{condition}"
    )


results_df = pd.DataFrame(
    all_rows
)

per_image_path = (
    RESULTS_DIR /
    "enhancement_evaluation_per_image.csv"
)

results_df.to_csv(
    per_image_path,
    index=False
)

print(
    "Saved:",
    per_image_path
)

display(
    results_df.head()
)

## 7. Aggregate metrics

The aggregate table separates validation and test results by condition. `PSNR/SSIM delta` shows the change relative to leaving the input untouched.

In [ ]:
if results_df.empty:
    raise RuntimeError(
        "No evaluation rows were produced. Verify that validation/test images "
        "exist and all three trained checkpoints loaded successfully."
    )

summary_rows = []

for (split, condition), group in results_df.groupby(
    ["split", "condition"]
):
    summary_rows.append({
        "split": split,
        "condition": condition,
        "images": len(group),

        "identity_psnr_mean":
            group["identity_psnr"].mean(),

        "enhanced_psnr_mean":
            group["enhanced_psnr"].mean(),

        "psnr_delta_mean":
            group["psnr_delta"].mean(),

        "identity_ssim_mean":
            group["identity_ssim"].mean(),

        "enhanced_ssim_mean":
            group["enhanced_ssim"].mean(),

        "ssim_delta_mean":
            group["ssim_delta"].mean(),

        "brightness_delta_mean":
            group["brightness_delta"].mean(),

        "contrast_delta_mean":
            group["contrast_delta"].mean(),

        "saturation_delta_mean":
            group["saturation_delta"].mean(),
    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULTS_DIR /
    "enhancement_evaluation_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(4)
)

print(
    "Saved:",
    summary_path
)

## 8. Test-set error analysis

Identify images where enhancement moves PSNR or SSIM substantially downward relative to the identity baseline. These cases are useful for manual inspection before using the enhancement models in the detector pipeline.

In [ ]:
test_results = results_df[
    results_df["split"] == "test"
].copy()

worst_psnr = (
    test_results
    .sort_values("psnr_delta")
    .groupby("condition")
    .head(20)
    .reset_index(drop=True)
)

worst_ssim = (
    test_results
    .sort_values("ssim_delta")
    .groupby("condition")
    .head(20)
    .reset_index(drop=True)
)

worst_psnr_path = (
    RESULTS_DIR /
    "enhancement_test_worst_psnr_cases.csv"
)

worst_ssim_path = (
    RESULTS_DIR /
    "enhancement_test_worst_ssim_cases.csv"
)

worst_psnr.to_csv(
    worst_psnr_path,
    index=False
)

worst_ssim.to_csv(
    worst_ssim_path,
    index=False
)

print(
    "Worst PSNR cases:",
    worst_psnr_path
)

print(
    "Worst SSIM cases:",
    worst_ssim_path
)

display(
    worst_psnr.head(10)
)

## 9. Visual comparison on real test images

For each condition, display:

1. original input,
2. enhanced output,
3. pseudo-reference.

These are real test images, but they are used only for evaluation/visualization and are never used to update model weights.

In [ ]:
def show_examples(
    condition,
    n=4,
    split="test"
):
    frame = evaluation_frames[
        (split, condition)
    ]

    if len(frame) == 0:
        print(
            f"No {split} images for {condition}."
        )
        return

    n = min(
        n,
        len(frame)
    )

    dataset = EvaluationDataset(
        frame.iloc[:n],
        REFERENCE_FUNCTIONS[condition]
    )

    model = models[condition]

    fig, axes = plt.subplots(
        n,
        3,
        figsize=(12, 4 * n)
    )

    if n == 1:
        axes = np.expand_dims(
            axes,
            axis=0
        )

    for i in range(n):
        x, y, path = dataset[i]

        with torch.no_grad():
            enhanced = model(
                x.unsqueeze(0).to(DEVICE)
            ).squeeze(0).cpu().clamp(
                0,
                1
            )

        axes[i, 0].imshow(
            x.permute(1, 2, 0)
        )
        axes[i, 0].set_title(
            f"{condition.title()} input"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(
            enhanced.permute(1, 2, 0)
        )
        axes[i, 1].set_title(
            "Enhanced"
        )
        axes[i, 1].axis("off")

        axes[i, 2].imshow(
            y.permute(1, 2, 0)
        )
        axes[i, 2].set_title(
            "Pseudo-reference"
        )
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


for condition in [
    "night",
    "rain",
    "fog"
]:
    show_examples(
        condition,
        n=MAX_VISUALS_PER_CONDITION,
        split="test"
    )

## 10. Enhancement magnitude analysis

Large brightness/contrast changes are not automatically good or bad. This section records the distribution of changes so overly aggressive models can be identified for manual review.

In [ ]:
for condition in [
    "night",
    "rain",
    "fog"
]:
    subset = test_results[
        test_results["condition"] == condition
    ]

    if len(subset) == 0:
        continue

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4)
    )

    axes[0].hist(
        subset["brightness_delta"],
        bins=30
    )
    axes[0].set_title(
        f"{condition.title()} brightness change"
    )
    axes[0].set_xlabel(
        "Enhanced - input"
    )

    axes[1].hist(
        subset["contrast_delta"],
        bins=30
    )
    axes[1].set_title(
        f"{condition.title()} contrast change"
    )
    axes[1].set_xlabel(
        "Enhanced - input"
    )

    plt.tight_layout()
    plt.show()

## 11. Save final evaluation report

The JSON report contains dataset counts, checkpoint paths, aggregate metrics, and the evaluation caveat about pseudo-reference targets.

In [ ]:
condition_counts = {}

for condition in [
    "night",
    "rain",
    "fog"
]:
    condition_counts[condition] = {}

    for split in [
        "val",
        "test"
    ]:
        condition_counts[
            condition
        ][split] = int(
            len(
                evaluation_frames[
                    (split, condition)
                ]
            )
        )

report = {
    "task": "enhancement_evaluation",
    "project_root": str(PROJECT_ROOT),
    "image_size": IMAGE_SIZE,
    "device": str(DEVICE),
    "checkpoints": {
        k: str(v)
        for k, v in CHECKPOINTS.items()
    },
    "condition_counts": condition_counts,
    "summary_csv": str(summary_path),
    "per_image_csv": str(per_image_path),
    "worst_psnr_csv": str(worst_psnr_path),
    "worst_ssim_csv": str(worst_ssim_path),
    "metric_note": (
        "PSNR/SSIM compare outputs with deterministic "
        "pseudo-reference targets matching the training "
        "notebooks. They are not human-annotated restoration "
        "ground truth."
    ),
    "system_level_note": (
        "Downstream object detection and tracking evaluation "
        "is required to determine whether enhancement improves "
        "the complete Adaptive_Road_Vision pipeline."
    ),
}

report_path = (
    RESULTS_DIR /
    "enhancement_evaluation_report.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2
    )

print(
    "Saved:",
    report_path
)

print(
    json.dumps(
        report,
        indent=2
    )
)

## 12. Output checklist

After successful execution, the main artifacts are:

```text
results/
└── enhancement/
    ├── enhancement_evaluation_per_image.csv
    ├── enhancement_evaluation_summary.csv
    ├── enhancement_test_worst_psnr_cases.csv
    ├── enhancement_test_worst_ssim_cases.csv
    └── enhancement_evaluation_report.json
```

The three trained checkpoints remain under:

```text
models/
└── enhancement/
    ├── night/night_enhancer_best.pth
    ├── rain/rain_enhancer_best.pth
    └── fog/fog_enhancer_best.pth
```

Notebook 17 is therefore the evaluation bridge before moving into detector training/comparison.